In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))

import ee
import geemap
from utils.variables import (
    PROJECT,
    COUNTRIES_ASSET_ID,
    PAS_ASSET_ID,
    OECMS_ASSET_ID,
    WKT_6933,
    PSM_CELL_SIZE,
)

import pandas as pd
import geopandas as gpd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

from absolute_effectiveness.site_selector import SiteSelector

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()

Part 1: Fit a global propensity model

In [ ]:
# Select AOI (country/region, eventually scale up to the whole world)

countries = ee.FeatureCollection(COUNTRIES_ASSET_ID)
aoi = countries.filter(ee.Filter.eq("country_na", "Ghana"))

In [ ]:
# Import PAs and filter to AOI

PAs = ee.FeatureCollection(PAS_ASSET_ID).filterBounds(aoi)
OECMS = ee.FeatureCollection(OECMS_ASSET_ID).filterBounds(aoi)

all_PAs = (
    ee.FeatureCollection([PAs, OECMS])
    .flatten()
    .filter(ee.Filter.eq("REALM", "Terrestrial"))
)

# Paint a binary PA mask (protected = 1, unprotected = 0)

EE_6933 = ee.Projection(WKT_6933).atScale(PSM_CELL_SIZE)  # equal area projection
protected_mask = (
    ee.Image.constant(0)
    .rename("protected")
    .paint(featureCollection=all_PAs, color=1)
    .reproject(crs=EE_6933)
)

In [ ]:
# Import covariate datasets (start with just elevation and slope, and then add more)

elevation = ee.ImageCollection("COPERNICUS/DEM/GLO30").filterBounds(aoi).select("DEM")
slope = elevation.map(lambda tile: ee.Terrain.slope(tile))

elevation = elevation.mosaic().rename("elevation")
slope = slope.mosaic()

# Resample covariates to 1km resolution
# For categorial covariates, use mode reducer

elevation = (
    elevation.setDefaultProjection(EE_6933)
    .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=4096)
    .reproject(protected_mask.projection())
)

slope = (
    slope.setDefaultProjection(EE_6933)
    .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=4096)
    .reproject(protected_mask.projection())
)

# Stack covariates in a single multi-band image
covariates = protected_mask.addBands(elevation).addBands(slope)

In [ ]:
# Take a stratified sample of covariates inside and outside PAs
# Can additionally stratify by biome, region, etc.

sample_pixels = covariates.stratifiedSample(
    region=aoi,
    scale=PSM_CELL_SIZE,
    numPoints=100,
    projection=EE_6933,
    seed=42,
    classBand="protected",
    classValues=[0, 1],
    classPoints=[100, 50],  # twice as many control points as treatment points
    dropNulls=True,
    geometries=True,  # remove geometries once we scale up and don't need to visualize anymore
)
# print(sample_pixels.first().getInfo())

# Convert table of sampled training data to dataframe
samples_list = sample_pixels.getInfo()["features"]
samples_df = pd.DataFrame([feature["properties"] for feature in samples_list])

print(samples_df.head())

In [ ]:
# Use training data to fit a propensity model (logistic regression)
X = samples_df[["elevation", "slope"]]
y = samples_df["protected"]
model = Pipeline(
    [("scaler", StandardScaler()), ("logit", LogisticRegression(max_iter=1000))]
)
model.fit(X, y)

# Examine coefficients to confirm that the data is reproducing the expected PA location bias
predictors = X.columns
logit = model.named_steps["logit"]
coef_df = pd.DataFrame({"variable": predictors, "coefficient": logit.coef_[0]})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df["effect_on_protection"] = coef_df["coefficient"].apply(
    lambda x: "positive" if x > 0 else "negative"
)
# print("\nCoefficient summary:")
# print(coef_df.sort_values("abs_coefficient", ascending=False))

In [ ]:
# TO DO:
#   maybe find some kind of land mask so we aren't sampling the ocean?
#   add more covariates
#   stratify sample by biome, region, etc.
#   test on larger AOIs, increase sample size


#   make sure control cells don't intersect any other PAs
#   figure out a more efficient grid sampling strategy for larger PAs
#   use all valid grids inside PA but randomly sample points outside PA?
#   enforce similarity between matched cells (biome, country, etc.)


Part 2: Match treatment and control cells

In [ ]:
# Load valid grid cells for a given PA

# Select PA
PA_ID = "1543"
site_id = 1543
test_sites = site_selector.get_test_sites()
site_geom = site_selector.get_site_geom(test_sites, site_id)

# Import grids
grid_gdf = gpd.read_parquet("../data/test_sites_TPA_PSM_GRID.parquet").to_crs(epsg=4326)

# Filter to grids for the selected PA
grid_gdf = grid_gdf[grid_gdf["WDPA_PID"] == PA_ID].copy()

# Convert to ee.FeatureCollection
grid_fc = geemap.geopandas_to_ee(grid_gdf)

# Add a unique cell_ID to each grid cell
cell_IDs = ee.List.sequence(0, grid_fc.size().getInfo() - 1)
featureList = grid_fc.toList(grid_fc.size())
grid_fc = ee.FeatureCollection(
    cell_IDs.map(
        lambda cell_ID: ee.Feature(featureList.get(cell_ID)).set(
            {"cell_ID": cell_ID, "label": None}
        )
    )
)

print(grid_fc.size().getInfo())
# This number is way too big (10,514), even for a small PA (385 km2)

In [ ]:
# To cut down the number of control cells, take a random sample

treatment_cells = grid_fc.filter(
    ee.Filter.eq("protected", 1)
)  # use all treatment cells
n_control = treatment_cells.size().getInfo() * 4
control_cells = (
    grid_fc.filter(ee.Filter.eq("protected", 0))
    .randomColumn(columnName="random", seed=43)
    .sort("random")
    .limit(n_control)
)

grid_fc = ee.FeatureCollection([treatment_cells, control_cells]).flatten()

# print(grid_fc.size().getInfo())
# This is a more manageable number (1,640). EE aborts at 5,000.

In [ ]:
# Aggregate covariates within grid cells
# will need to use mode reducer for categorical covariates

grid_fc = (
    covariates.select("elevation", "slope")
    .reduceRegions(collection=grid_fc, reducer=ee.Reducer.mean(), scale=PSM_CELL_SIZE)
    .select("cell_ID", "elevation", "slope", "protected")
)

# print(grid_fc.first().getInfo())

In [ ]:
# Apply global propensity model from Part 1 to predict propensity scores for each grid cell

grid_list = grid_fc.getInfo()["features"]
grid_df = pd.DataFrame([feature["properties"] for feature in grid_list])

X = grid_df[["elevation", "slope"]]
grid_df["propensity_score"] = model.predict_proba(X)[:, 1]

print(grid_df.head())

In [ ]:
# Match each treatment cell to up to 4 control cells
# Re-using control cells is ok

# Split treatment and control
treat_df = grid_df[grid_df["protected"] == 1].copy().reset_index(drop=True)
control_df = grid_df[grid_df["protected"] == 0].copy().reset_index(drop=True)

# Convert to 2D array
X_control = control_df[["propensity_score"]].to_numpy()
X_treat = treat_df[["propensity_score"]].to_numpy()

# Fit nearest-neighbor search on control cells
nn = NearestNeighbors(
    n_neighbors=4,
    metric="euclidean",  # in 1D, this is just absolute difference
)
nn.fit(X_control)

# Find the 4 nearest control cells for each treatment cell
distances, indices = nn.kneighbors(X_treat)

# Build a long-form match table
matches = []
caliper = 0.2

for i in range(len(treat_df)):
    treat_row = treat_df.iloc[i]
    treat_id = treat_row["cell_ID"]
    treat_score = treat_row["propensity_score"]

    for rank, (dist, j) in enumerate(zip(distances[i], indices[i]), start=1):
        # Apply caliper
        if dist <= caliper:
            control_row = control_df.iloc[j]

            matches.append(
                {
                    "treat_cell_id": treat_id,
                    "control_cell_id": control_row["cell_ID"],
                    "treat_score": treat_score,
                    "control_score": control_row["propensity_score"],
                    "ps_distance": float(dist),
                    "match_rank": rank,
                }
            )

match_df = pd.DataFrame(matches).sort_values(by="treat_cell_id")

print(match_df.head())
print(f"\nNumber of matched treatment cells: {match_df['treat_cell_id'].nunique()}")
print(f"Total matched pairs: {len(match_df)}")
unmatched_treat_ids = set(treat_df["cell_ID"]) - set(match_df["treat_cell_id"])
print(f"Unmatched treatment cells: {len(unmatched_treat_ids)}")

In [ ]:
# Filter grid cells to only valid matches

valid_ids = pd.concat([match_df["treat_cell_id"], match_df["control_cell_id"]]).unique()
valid_ids = ee.List(valid_ids.astype(int).tolist())
matched_grids = grid_fc.filter(ee.Filter.inList("cell_ID", valid_ids))

In [ ]:
print(samples_df.head())
print("\nCoefficient summary:")
print(coef_df.sort_values("abs_coefficient", ascending=False))

In [ ]:
Map = geemap.Map()

Map.add_basemap("CartoDB.Positron")

Map.addLayer(aoi, {}, "AOI")
Map.centerObject(aoi)

# Propensity Model
Map.addLayer(
    covariates.select("protected").clip(aoi),
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "Protected",
)
Map.addLayer(
    covariates.select("elevation"),
    {"min": 0, "max": 800, "palette": ["blue", "green", "yellow", "red"]},
    "Elevation",
)
Map.addLayer(
    covariates.select("slope"),
    {"min": 0, "max": 12, "palette": ["blue", "green", "yellow", "red"]},
    "Slope",
)
Map.addLayer(sample_pixels, {"color": "red"}, "Sample Pixels")

# Cell Matching
Map.addLayer(site_geom, {"color": "red"}, "Site")
Map.addLayer(grid_fc, {}, "Grid")
Map.addLayer(matched_grids, {}, "Matched Cells")

Map